<a href="https://colab.research.google.com/github/beabritw/datasus-dengue-social/blob/feature%2Ftransform/transform.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q pyspark pandas pyarrow

In [2]:
from google.colab import drive
import os
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, when, count, isnan, trim, upper,
    regexp_extract, substring, cast
)
from pyspark.sql import functions as F
import pandas as pd

In [3]:
drive.mount('/content/drive')

base_path      = '/content/drive/MyDrive/Topicos_BD'
raw_sinan      = os.path.join(base_path, 'raw', 'sinan', 'sinan_dengue_raw')
raw_ibge       = os.path.join(base_path, 'raw', 'ibge', 'ibge_municipios.parquet')
processed_path = os.path.join(base_path, 'processed')

os.makedirs(processed_path, exist_ok=True)

print('Caminhos configurados:')
print(f'  raw/sinan -> {raw_sinan}')
print(f'  raw/ibge  -> {raw_ibge}')
print(f'  processed -> {processed_path}')

Mounted at /content/drive
Caminhos configurados:
  raw/sinan -> /content/drive/MyDrive/Topicos_BD/raw/sinan/sinan_dengue_raw
  raw/ibge  -> /content/drive/MyDrive/Topicos_BD/raw/ibge/ibge_municipios.parquet
  processed -> /content/drive/MyDrive/Topicos_BD/processed


In [4]:
spark = SparkSession.builder \
    .appName('DataMajor_Transform') \
    .config('spark.driver.memory', '8g') \
    .getOrCreate()

print(f'Spark versão: {spark.version}')

Spark versão: 4.0.2


In [5]:
caminho_bruto_sinan = '/content/drive/MyDrive/Topicos_BD/raw/sinan/DENGBR*.parquet'

print("Lendo os arquivos originais do PySUS...")
df_sinan = spark.read.parquet(caminho_bruto_sinan)

COLUNAS_INTERESSE = [
    'HOSPITALIZ',
    'CS_SEXO', 'NU_IDADE_N', 'ID_MUNICIP', 'ID_OCUPA_N', 'NU_ANO',
    'CS_RACA', 'CS_ESCOL_N',
    'FEBRE', 'MIALGIA', 'CEFALEIA', 'VOMITO', 'NAUSEA',
    'DOR_RETRO', 'DOR_COSTAS', 'EXANTEMA', 'LEUCOPENIA',
    'DIABETES', 'HIPERTENSA', 'RENAL', 'HEPATOPAT', 'HEMATOLOG',
    'CLASSI_FIN', 'EVOLUCAO'
]

df_sinan = df_sinan.select(COLUNAS_INTERESSE)

df_ibge = pd.read_parquet('/content/drive/MyDrive/Topicos_BD/raw/ibge/ibge_municipios.parquet')

print(f'[SINAN] Registros: {df_sinan.count():,}  |  Colunas: {len(df_sinan.columns)}')
print(f'[IBGE]  Municípios: {len(df_ibge):,}')

Lendo os arquivos originais do PySUS...
[SINAN] Registros: 9,615,975  |  Colunas: 24
[IBGE]  Municípios: 5,571



Lê de: `raw/`  
Salva em: `processed/`

In [6]:
# Visão inicial do schema SINAN
df_sinan.printSchema()

root
 |-- HOSPITALIZ: string (nullable = true)
 |-- CS_SEXO: string (nullable = true)
 |-- NU_IDADE_N: string (nullable = true)
 |-- ID_MUNICIP: string (nullable = true)
 |-- ID_OCUPA_N: string (nullable = true)
 |-- NU_ANO: string (nullable = true)
 |-- CS_RACA: string (nullable = true)
 |-- CS_ESCOL_N: string (nullable = true)
 |-- FEBRE: string (nullable = true)
 |-- MIALGIA: string (nullable = true)
 |-- CEFALEIA: string (nullable = true)
 |-- VOMITO: string (nullable = true)
 |-- NAUSEA: string (nullable = true)
 |-- DOR_RETRO: string (nullable = true)
 |-- DOR_COSTAS: string (nullable = true)
 |-- EXANTEMA: string (nullable = true)
 |-- LEUCOPENIA: string (nullable = true)
 |-- DIABETES: string (nullable = true)
 |-- HIPERTENSA: string (nullable = true)
 |-- RENAL: string (nullable = true)
 |-- HEPATOPAT: string (nullable = true)
 |-- HEMATOLOG: string (nullable = true)
 |-- CLASSI_FIN: string (nullable = true)
 |-- EVOLUCAO: string (nullable = true)



In [7]:
# Amostra
df_sinan.show(10, truncate=False)

+----------+-------+----------+----------+----------+------+-------+----------+-----+-------+--------+------+------+---------+----------+--------+----------+--------+----------+-----+---------+---------+----------+--------+
|HOSPITALIZ|CS_SEXO|NU_IDADE_N|ID_MUNICIP|ID_OCUPA_N|NU_ANO|CS_RACA|CS_ESCOL_N|FEBRE|MIALGIA|CEFALEIA|VOMITO|NAUSEA|DOR_RETRO|DOR_COSTAS|EXANTEMA|LEUCOPENIA|DIABETES|HIPERTENSA|RENAL|HEPATOPAT|HEMATOLOG|CLASSI_FIN|EVOLUCAO|
+----------+-------+----------+----------+----------+------+-------+----------+-----+-------+--------+------+------+---------+----------+--------+----------+--------+----------+-----+---------+---------+----------+--------+
|2         |F      |4016      |210780    |999991    |2024  |4      |5         |1    |1      |1       |1     |1     |1        |1         |2       |2         |2       |2         |2    |2        |2        |10        |1       |
|          |F      |4020      |211220    |          |2024  |4      |9         |1    |1      |2       |1 

---
### 2. Variável alvo — `HOSPITALIZ`

Esta é a coluna mais importante do projeto. Distribuição encontrada no Extract:

| Valor | Significado      | Quantidade | Ação                         |
|-------|-----------------|------------|------------------------------|
| `2`   | Não hospitalizado | 4.295.622 | Manter → label **0**        |
| `1`   | Hospitalizado     | 206.578   | Manter → label **1**        |
| `''`  | Vazio             | 1.457.862 | Imputar via `CLASSI_FIN`    |
| `9`   | Ignorado          | 142.032   | **Remover**                  |
| `0`   | Inválido          | 8         | **Remover**                  |

**Decisão adotada para os 1.4M vazios:**  
Cruzamento com `CLASSI_FIN`: se `CLASSI_FIN ∈ {11, 12}` (Dengue com sinais de alarme ou Dengue grave), infere-se hospitalização (`label = 1`). Caso contrário, o registro é removido. Essa abordagem preserva mais dados do que simplesmente descartar os 24% vazios, com embasamento clínico razoável.

---

In [8]:
# Distribuição atual de HOSPITALIZ antes do tratamento
print('Distribuição de HOSPITALIZ (antes):')
df_sinan.groupBy('HOSPITALIZ').count().orderBy('count', ascending=False).show()

Distribuição de HOSPITALIZ (antes):
+----------+-------+
|HOSPITALIZ|  count|
+----------+-------+
|         2|6595023|
|          |2494140|
|         1| 293541|
|         9| 233264|
|         0|      7|
+----------+-------+



In [9]:
# Remover registros inválidos (HOSPITALIZ = '9' ou '0')
df = df_sinan.filter(
    ~col('HOSPITALIZ').isin(['9', '0'])
)
print(f'Após remover 9 e 0: {df.count():,} registros')

Após remover 9 e 0: 9,382,704 registros


In [10]:
# Imputar vazios com base em CLASSI_FIN
# CLASSI_FIN = 11 (Dengue com sinais de alarme) ou 12 (Dengue grave) → hospitalizado = 1
# Demais vazios → remover

df = df.withColumn(
    'HOSPITALIZ',
    when(
        (trim(col('HOSPITALIZ')) == '') & col('CLASSI_FIN').isin(['11', '12']),
        '1'
    ).otherwise(col('HOSPITALIZ'))
)

# Remove registros que ainda têm HOSPITALIZ vazio após a imputação
df = df.filter(trim(col('HOSPITALIZ')) != '')

print(f'Após imputação e remoção de vazios restantes: {df.count():,} registros')

Após imputação e remoção de vazios restantes: 6,901,591 registros


In [11]:
# Converter para label binário: '1' → 1, '2' → 0
df = df.withColumn(
    'HOSPITALIZ',
    when(col('HOSPITALIZ') == '1', 1)
    .when(col('HOSPITALIZ') == '2', 0)
    .otherwise(None)
).filter(col('HOSPITALIZ').isNotNull())

print('Distribuição de HOSPITALIZ (após tratamento):')
df.groupBy('HOSPITALIZ').count().show()
print(f'Total de registros: {df.count():,}')

Distribuição de HOSPITALIZ (após tratamento):
+----------+-------+
|HOSPITALIZ|  count|
+----------+-------+
|         1| 306568|
|         0|6595023|
+----------+-------+

Total de registros: 6,901,591


Remover casos descartados — `CLASSI_FIN = 5`

Registros com `CLASSI_FIN = 5` são casos descartados de dengue — não devem fazer parte do dataset de treinamento.

In [12]:
antes = df.count()
df = df.filter(col('CLASSI_FIN') != '5')
depois = df.count()

print(f'Registros removidos (CLASSI_FIN = 5): {antes - depois:,}')
print(f'Registros restantes: {depois:,}')

Registros removidos (CLASSI_FIN = 5): 0
Registros restantes: 6,901,591


Campos binários — padronização `1/0`

Todos os sinais clínicos e comorbidades chegam do SINAN com `1 = Sim` e `2 = Não` (strings). Para o K-means e demais modelos, precisam ser convertidos para `1 = Sim` e `0 = Não`.

Os valores chegam como **string** no Spark. Use `col(c) == '1'`, não `col(c) == 1`.

In [13]:
colunas_binarias = [
    'FEBRE', 'MIALGIA', 'CEFALEIA', 'VOMITO', 'NAUSEA',
    'DOR_RETRO', 'DOR_COSTAS', 'EXANTEMA', 'LEUCOPENIA',
    'DIABETES', 'HIPERTENSA', 'RENAL', 'HEPATOPAT', 'HEMATOLOG',
]

for c in colunas_binarias:
    df = df.withColumn(
        c,
        when(col(c) == '1', 1)
        .when(col(c) == '2', 0)
        .otherwise(None)  # string vazia, '9', outros → None
    )

print('Campos binários padronizados. Amostra de valores:')
df.select(colunas_binarias[:5]).show(5)

Campos binários padronizados. Amostra de valores:
+-----+-------+--------+------+------+
|FEBRE|MIALGIA|CEFALEIA|VOMITO|NAUSEA|
+-----+-------+--------+------+------+
|    1|      1|       1|     1|     1|
|    1|      1|       1|     1|     1|
|    0|      1|       1|     1|     1|
|    1|      1|       1|     0|     0|
|    1|      1|       1|     1|     1|
+-----+-------+--------+------+------+
only showing top 5 rows


In [14]:
# 1. Tratamento Inteligente de Dados Sociais (Evitando Viés de Seleção)
colunas_sociais = ['CS_RACA', 'CS_ESCOL_N']

for c in colunas_sociais:
    if c in df.columns:
        df = df.withColumn(
            c,
            when(
                col(c).isNull() |
                (trim(col(c)) == '') |
                (col(c) == '9') |
                (col(c) == '0'),
                'Ausente/Nao_Coletado'
            ).otherwise(col(c))
        )

Campo de idade — `NU_IDADE_N`

O SINAN usa codificação especial: o 1º dígito indica a unidade (1 = Horas; 2 = Dias; 3 = Meses; 4 = Anos)
Exemplos: `4025` = 25 anos | `3006` = 6 meses.

A transformação extrai somente registros em anos. Recém-nascidos e casos em meses/dias recebem `idade_anos = 0`.

In [15]:
df = df.withColumn(
    'idade_anos',
    when(
        col('NU_IDADE_N').startswith('4'),
        col('NU_IDADE_N').substr(2, 3).cast('int')
    ).otherwise(0)
)

df = df.filter(
    (col('idade_anos') >= 0) & (col('idade_anos') <= 120)
)

df = df.drop('NU_IDADE_N')

print('Distribuição corrigida de idade_anos (estatísticas básicas):')
df.select('idade_anos').describe().show()

Distribuição corrigida de idade_anos (estatísticas básicas):
+-------+-----------------+
|summary|       idade_anos|
+-------+-----------------+
|  count|          6901376|
|   mean|35.91849900657492|
| stddev|20.10073670416327|
|    min|                0|
|    max|              120|
+-------+-----------------+



Join SINAN × IBGE

O IBGE enriquece os dados com `nome_municipio`, `uf_sigla`, `uf_nome` e `regiao`.  
O cruzamento é feito por código de município — porém os dois sistemas usam tamanhos diferentes:

| Sistema | Campo              | Exemplo   | Dígitos |
|---------|--------------------|-----------|---------|
| SINAN   | `ID_MUNICIP`       | `530010`  | 6       |
| IBGE    | `id_municipio`     | `5300108` | 7       |
| IBGE    | `id_municipio_sinan` | `530010` | 6 (já calculado no Extract) |

**left join** para não perder registros do SINAN sem correspondência no IBGE.


In [16]:
# Converte o DataFrame IBGE (pandas) para Spark
df_ibge_spark = spark.createDataFrame(df_ibge)

# Left join pelo código de município (6 dígitos)
df = df.join(
    df_ibge_spark,
    df.ID_MUNICIP == df_ibge_spark.id_municipio_sinan,
    how='left'
)

#Remove colunas redundantes do IBGE que não agregam valor
colunas_remover = ['id_municipio', 'id_municipio_sinan', 'uf_codigo']
df = df.drop(*colunas_remover)

print(f'Registros após join: {df.count():,}')
print(f'Colunas após join: {len(df.columns)}')
print('\nColunas adicionadas do IBGE:')
print([c for c in df.columns if c in ['nome_municipio', 'uf_sigla', 'uf_nome', 'regiao']])

Registros após join: 6,901,376
Colunas após join: 28

Colunas adicionadas do IBGE:
['nome_municipio', 'uf_sigla', 'uf_nome', 'regiao']


In [17]:
# Verifica quantos registros não encontraram correspondência no IBGE
sem_municipio = df.filter(col('nome_municipio').isNull()).count()
print(f'Registros sem correspondência no IBGE: {sem_municipio:,}')

Registros sem correspondência no IBGE: 0


Verificação e tratamento de nulos

Colunas com alta proporção de nulos prejudicam o K-means e a classificação. A política adotada:
- Colunas com **> 30% de nulos** são removidas do dataset (dados insuficientes para inferência confiável).
- Colunas binárias (`sinais clínicos / comorbidades`) com nulos moderados: preenchimento com `0` (ausência do sintoma/comorbidade — abordagem conservadora).
- Demais colunas: remoção das linhas nulas.

In [18]:
# Conta nulos por coluna e calcula percentual
total = df.count()

nulos_df = df.select([
    count(when(col(c).isNull(), c)).alias(c)
    for c in df.columns
]).toPandas().T

nulos_df.columns = ['nulos']
nulos_df['pct'] = (nulos_df['nulos'] / total * 100).round(2)
nulos_df = nulos_df.sort_values('pct', ascending=False)

print('Nulos por coluna (ordenado por %):')
print(nulos_df[nulos_df['nulos'] > 0].to_string())

Nulos por coluna (ordenado por %):
            nulos  pct
DIABETES      211  0.0
HIPERTENSA    205  0.0
RENAL         202  0.0
HEPATOPAT     204  0.0
HEMATOLOG     201  0.0


In [19]:
# Remove colunas com mais de 30% de nulos
LIMITE_NULOS_PCT = 30.0

colunas_alta_nulidade = nulos_df[nulos_df['pct'] > LIMITE_NULOS_PCT].index.tolist()

if colunas_alta_nulidade:
    print(f'Colunas removidas por alta nulidade (> {LIMITE_NULOS_PCT}%): {colunas_alta_nulidade}')
    df = df.drop(*colunas_alta_nulidade)
else:
    print('Nenhuma coluna com alta nulidade encontrada.')

Nenhuma coluna com alta nulidade encontrada.


In [20]:
# Preenche nulos em colunas binárias com 0 (ausência do sintoma/comorbidade)
binarias_presentes = [c for c in colunas_binarias if c in df.columns]

df = df.fillna(0, subset=binarias_presentes)

print(f'Nulos nas colunas binárias preenchidos com 0 ({len(binarias_presentes)} colunas).')

Nulos nas colunas binárias preenchidos com 0 (14 colunas).


In [21]:
# Remove linhas com nulos nas colunas essenciais restantes
colunas_essenciais = ['HOSPITALIZ', 'CS_SEXO', 'idade_anos', 'ID_MUNICIP']
colunas_essenciais_presentes = [c for c in colunas_essenciais if c in df.columns]

antes = df.count()
df = df.dropna(subset=colunas_essenciais_presentes)
depois = df.count()

print(f'Linhas removidas por nulos em colunas essenciais: {antes - depois:,}')
print(f'Registros restantes: {depois:,}')

Linhas removidas por nulos em colunas essenciais: 0
Registros restantes: 6,901,376


Conversão de tipos

No Spark, colunas do SINAN chegam como `string`. Precisam ser convertidas para seus tipos corretos.

In [22]:
# 1. Tratamento da coluna CS_SEXO
df = df.withColumn('CS_SEXO', trim(upper(col('CS_SEXO'))))

# 2. Tratamento e cast seguro de NU_ANO
df = df.withColumn(
    'NU_ANO',
    when(trim(col('NU_ANO')) == '', None)
    .otherwise(col('NU_ANO'))
    .cast('int')
)

# 3. Tratamento e cast seguro de EVOLUCAO
if 'EVOLUCAO' in df.columns:
    df = df.withColumn(
        'EVOLUCAO',
        when(col('EVOLUCAO').isin(['9']) | (trim(col('EVOLUCAO')) == ''), None)
        .otherwise(col('EVOLUCAO'))
        .cast('int')
    )

# 4. Tratamento e cast seguro de CLASSI_FIN
if 'CLASSI_FIN' in df.columns:
    df = df.withColumn(
        'CLASSI_FIN',
        when(trim(col('CLASSI_FIN')) == '', None)
        .otherwise(col('CLASSI_FIN'))
        .cast('int')
    )

print('Schema após conversão segura de tipos:')
df.printSchema()

Schema após conversão segura de tipos:
root
 |-- HOSPITALIZ: integer (nullable = true)
 |-- CS_SEXO: string (nullable = true)
 |-- ID_MUNICIP: string (nullable = true)
 |-- ID_OCUPA_N: string (nullable = true)
 |-- NU_ANO: integer (nullable = true)
 |-- CS_RACA: string (nullable = true)
 |-- CS_ESCOL_N: string (nullable = true)
 |-- FEBRE: integer (nullable = true)
 |-- MIALGIA: integer (nullable = true)
 |-- CEFALEIA: integer (nullable = true)
 |-- VOMITO: integer (nullable = true)
 |-- NAUSEA: integer (nullable = true)
 |-- DOR_RETRO: integer (nullable = true)
 |-- DOR_COSTAS: integer (nullable = true)
 |-- EXANTEMA: integer (nullable = true)
 |-- LEUCOPENIA: integer (nullable = true)
 |-- DIABETES: integer (nullable = true)
 |-- HIPERTENSA: integer (nullable = true)
 |-- RENAL: integer (nullable = true)
 |-- HEPATOPAT: integer (nullable = true)
 |-- HEMATOLOG: integer (nullable = true)
 |-- CLASSI_FIN: integer (nullable = true)
 |-- EVOLUCAO: integer (nullable = true)
 |-- idade_ano

In [23]:
# Considerando 'M' (Masculino), 'F' (Feminino) e 'I' ou 'Ausente'
if 'CS_SEXO' in df.columns:
    df = df.withColumn('is_mulher', when(col('CS_SEXO') == 'F', 1).otherwise(0))
    df = df.withColumn('is_homem', when(col('CS_SEXO') == 'M', 1).otherwise(0))
    df = df.withColumn('is_sexo_ausente', when(~col('CS_SEXO').isin(['F', 'M']), 1).otherwise(0))

# Dicionário SINAN: 1-Branca, 2-Preta, 3-Amarela, 4-Parda, 5-Indígena
if 'CS_RACA' in df.columns:
    df = df.withColumn('is_branco', when(col('CS_RACA') == '1', 1).otherwise(0))
    df = df.withColumn('is_preto', when(col('CS_RACA') == '2', 1).otherwise(0))
    df = df.withColumn('is_amarelo', when(col('CS_RACA') == '3', 1).otherwise(0))
    df = df.withColumn('is_pardo', when(col('CS_RACA') == '4', 1).otherwise(0))
    df = df.withColumn('is_indigena', when(col('CS_RACA') == '5', 1).otherwise(0))
    df = df.withColumn('is_raca_ausente', when(col('CS_RACA') == 'Ausente/Nao_Coletado', 1).otherwise(0))

# (Escolaridade)
# Dicionário SINAN: 0-Analfabeto, 1 a 4-Fundamental, 5-Medio, 6-Superior
if 'CS_ESCOL_N' in df.columns:
    # Agrupando algumas categorias para não criar colunas demais (opcional, mas recomendado)
    df = df.withColumn('is_analfabeto', when(col('CS_ESCOL_N') == '0', 1).otherwise(0))
    df = df.withColumn('is_fundamental', when(col('CS_ESCOL_N').isin(['1', '2', '3', '4']), 1).otherwise(0))
    df = df.withColumn('is_medio', when(col('CS_ESCOL_N') == '5', 1).otherwise(0))
    df = df.withColumn('is_superior', when(col('CS_ESCOL_N') == '6', 1).otherwise(0))
    df = df.withColumn('is_escolaridade_ausente', when(col('CS_ESCOL_N') == 'Ausente/Nao_Coletado', 1).otherwise(0))

if 'regiao' in df.columns:
    df = df.withColumn('is_sudeste', when(col('regiao') == 'Sudeste', 1).otherwise(0))
    df = df.withColumn('is_nordeste', when(col('regiao') == 'Nordeste', 1).otherwise(0))
    df = df.withColumn('is_sul', when(col('regiao') == 'Sul', 1).otherwise(0))
    df = df.withColumn('is_norte', when(col('regiao') == 'Norte', 1).otherwise(0))
    df = df.withColumn('is_centro_oeste', when(col('regiao') == 'Centro-Oeste', 1).otherwise(0))

# dropa as colunas originais de texto, já que o K-Means só quer os números
colunas_originais_texto = ['CS_SEXO', 'CS_RACA', 'CS_ESCOL_N']
df = df.drop(*[c for c in colunas_originais_texto if c in df.columns])

print("Colunas binárias criadas com sucesso!")

Colunas binárias criadas com sucesso!


In [24]:
# from pyspark.ml.feature import StringIndexer, OneHotEncoder
# from pyspark.ml import Pipeline

# print("Iniciando conversão categórica (One-Hot Encoding) para o K-Means...")

# # Variáveis categóricas que precisam virar binárias
# cols_para_ohe = ['CS_SEXO', 'CS_RACA', 'CS_ESCOL_N']
# cols_presentes_ohe = [c for c in cols_para_ohe if c in df.columns]

# # Transforma a string em um índice numérico
# indexers = [
#     StringIndexer(inputCol=c, outputCol=f"{c}_idx", handleInvalid="keep")
#     for c in cols_presentes_ohe
# ]

# # Transforma o índice numérico em vetores binários (Dummy)
# encoders = [
#     OneHotEncoder(inputCol=f"{c}_idx", outputCol=f"{c}_vec", dropLast=False)
#     for c in cols_presentes_ohe
# ]

# # Executa as transformações
# pipeline = Pipeline(stages=indexers + encoders)
# modelo_pipeline = pipeline.fit(df)
# df = modelo_pipeline.transform(df)

# # Remove as colunas de índice temporárias para não sujar o dataset
# colunas_idx = [f"{c}_idx" for c in cols_presentes_ohe]
# df = df.drop(*colunas_idx)

# print("One-Hot Encoding concluído. Vetores criados:")
# print([f"{c}_vec" for c in cols_presentes_ohe])

In [25]:
colunas_para_dropar = [
    'CLASSI_FIN',
    'EVOLUCAO',

    'ID_MUNICIP',
    'nome_municipio',
    'uf_sigla',
    'uf_nome',
    'regiao',

    'ID_OCUPA_N',
    'NU_ANO'
]

# Realiza o drop apenas das colunas que ainda existirem no DataFrame
df = df.drop(*[c for c in colunas_para_dropar if c in df.columns])

print("Limpeza estrutural concluída! Colunas mantidas para o modelo:")
print(df.columns)

Limpeza estrutural concluída! Colunas mantidas para o modelo:
['HOSPITALIZ', 'FEBRE', 'MIALGIA', 'CEFALEIA', 'VOMITO', 'NAUSEA', 'DOR_RETRO', 'DOR_COSTAS', 'EXANTEMA', 'LEUCOPENIA', 'DIABETES', 'HIPERTENSA', 'RENAL', 'HEPATOPAT', 'HEMATOLOG', 'idade_anos', 'is_mulher', 'is_homem', 'is_sexo_ausente', 'is_branco', 'is_preto', 'is_amarelo', 'is_pardo', 'is_indigena', 'is_raca_ausente', 'is_analfabeto', 'is_fundamental', 'is_medio', 'is_superior', 'is_escolaridade_ausente', 'is_sudeste', 'is_nordeste', 'is_sul', 'is_norte', 'is_centro_oeste']


Relatório final do Transform


In [26]:
total_final = df.count()
total_original = 6_102_102

print()
print('Resumo doTransform')
print('=' * 52)
print(f'Registros originais (Extract): {total_original:>12,}')
print(f'Registros após Transform: {total_final:>12,}')
print(f'Registros removidos: {total_original - total_final:>12,}')
print(f'Retenção: {total_final/total_original*100:>11.1f}%')
print()
print('HOSPITALIZ = 9 e 0 removidos')
print('Vazios em HOSPITALIZ imputados via CLASSI_FIN {11, 12}')
print('HOSPITALIZ convertido para label binário 0/1')
print('CLASSI_FIN = 5 (descartados) removidos')
print('Campos binários padronizados: "2"→0, "1"→1')
print('NU_IDADE_N convertido para idade_anos')
print('Idades inválidas removidas (< 0 ou > 120)')
print('Join SINAN × IBGE por ID_MUNICIP (left join)')
print('Colunas com > 30% de nulos removidas')
print('Nulos em colunas binárias preenchidos com 0')
print('Linhas com nulos em colunas essenciais removidas')
print('Tipos convertidos (int, string normalizada)')
print()
print(f'  Colunas no dataset final: {len(df.columns)}')
print('=' * 52)

print('\nDistribuição final da variável alvo HOSPITALIZ:')
df.groupBy('HOSPITALIZ').count() \
  .withColumn('pct', F.round(col('count') / total_final * 100, 2)) \
  .orderBy('HOSPITALIZ') \
  .show()

if 'regiao' in df.columns:
    print('Distribuição por região:')
    df.groupBy('regiao').count().orderBy('count', ascending=False).show()


Resumo doTransform
Registros originais (Extract):    6,102,102
Registros após Transform:    6,901,376
Registros removidos:     -799,274
Retenção:       113.1%

HOSPITALIZ = 9 e 0 removidos
Vazios em HOSPITALIZ imputados via CLASSI_FIN {11, 12}
HOSPITALIZ convertido para label binário 0/1
CLASSI_FIN = 5 (descartados) removidos
Campos binários padronizados: "2"→0, "1"→1
NU_IDADE_N convertido para idade_anos
Idades inválidas removidas (< 0 ou > 120)
Join SINAN × IBGE por ID_MUNICIP (left join)
Colunas com > 30% de nulos removidas
Nulos em colunas binárias preenchidos com 0
Linhas com nulos em colunas essenciais removidas
Tipos convertidos (int, string normalizada)

  Colunas no dataset final: 35

Distribuição final da variável alvo HOSPITALIZ:
+----------+-------+-----+
|HOSPITALIZ|  count|  pct|
+----------+-------+-----+
|         0|6594826|95.56|
|         1| 306550| 4.44|
+----------+-------+-----+



Salvar resultado em `processed/`

In [27]:
output_path = os.path.join(processed_path, 'sinan_dengue_processed')

df.write.mode('overwrite').parquet(output_path)

print(f'Dataset processado salvo em:')
print(f'  {output_path}')
print(f'\nRegistros salvos: {total_final:,}')
print(f'Colunas salvas  : {len(df.columns)}')

Dataset processado salvo em:
  /content/drive/MyDrive/Topicos_BD/processed/sinan_dengue_processed

Registros salvos: 6,901,376
Colunas salvas  : 35


In [28]:
# Verificação: relê o parquet salvo e confirma integridade
df_check = spark.read.parquet(output_path)

print('Verificação do arquivo salvo:')
print(f'  Registros : {df_check.count():,}')
print(f'  Colunas   : {len(df_check.columns)}')
print()
df_check.show(3)
spark.stop()

Verificação do arquivo salvo:
  Registros : 6,901,376
  Colunas   : 35

+----------+-----+-------+--------+------+------+---------+----------+--------+----------+--------+----------+-----+---------+---------+----------+---------+--------+---------------+---------+--------+----------+--------+-----------+---------------+-------------+--------------+--------+-----------+-----------------------+----------+-----------+------+--------+---------------+
|HOSPITALIZ|FEBRE|MIALGIA|CEFALEIA|VOMITO|NAUSEA|DOR_RETRO|DOR_COSTAS|EXANTEMA|LEUCOPENIA|DIABETES|HIPERTENSA|RENAL|HEPATOPAT|HEMATOLOG|idade_anos|is_mulher|is_homem|is_sexo_ausente|is_branco|is_preto|is_amarelo|is_pardo|is_indigena|is_raca_ausente|is_analfabeto|is_fundamental|is_medio|is_superior|is_escolaridade_ausente|is_sudeste|is_nordeste|is_sul|is_norte|is_centro_oeste|
+----------+-----+-------+--------+------+------+---------+----------+--------+----------+--------+----------+-----+---------+---------+----------+---------+--------+----